# Dask delayed: build a task graph

**Core notebook, about 15 minutes.** We will turn separate file operations into tasks, inspect Dask's plan, and calculate the final result once.

In [ ]:
from pathlib import Path
import shutil
import time

import dask
from dask import delayed

data_dir = Path("temp_data")
data_dir.mkdir(exist_ok=True)
for index in range(5):
    lines = [f"sample {index} line {line}\n" for line in range(index + 1)]
    (data_dir / f"file_{index}.txt").write_text("".join(lines), encoding="utf-8")

## 1. Start with one normal function

Keep task functions small and testable. The short sleep represents file or network latency.

In [ ]:
def count_file(path):
    text = Path(path).read_text(encoding="utf-8")
    time.sleep(0.2)
    return {
        "path": str(path),
        "words": len(text.split()),
        "lines": len(text.splitlines()),
    }

count_file(data_dir / "file_0.txt")

## 2. Delay calls, then compute once

Calling a delayed function creates a task description. It does not immediately read the file.

In [ ]:
paths = sorted(data_dir.glob("*.txt"))
file_tasks = [delayed(count_file)(path) for path in paths]
file_tasks

In [ ]:
results = dask.compute(*file_tasks, scheduler="threads", num_workers=4)
results

## Your turn: put the final summary in the task graph

**5 minutes.** Create one delayed result containing total words and total lines across all files. Compute only the final summary.

Hint: write a normal `summarize(results)` function, then call it with `delayed(summarize)(file_tasks)`.

In [ ]:
# Write your solution here before running the solution cell.

### Solution and correctness check

Run the next cell after attempting the final summary. It checks the file, word, and line counts.

In [ ]:
def summarize(results):
    return {
        "files": len(results),
        "words": sum(item["words"] for item in results),
        "lines": sum(item["lines"] for item in results)
    }

summary_task = delayed(summarize)(file_tasks)
summary = summary_task.compute(scheduler="threads", num_workers=4)
assert summary == {"files": 5, "words": 60, "lines": 15}
summary

## 3. Inspect the graph

Each file can be processed independently. The summary waits for all five results.

In [ ]:
summary_task.visualize(filename="delayed_graph", format="svg")

In [ ]:
shutil.rmtree(data_dir)
Path("delayed_graph.svg").unlink(missing_ok=True)

## Takeaway

Make each task return a result instead of changing shared data. Keep all required steps in the task graph, then call `compute()` once when you need the final answer.